# Phase 3 · S1a — Full-ranking evaluation + re-eval checkpoints

**Mục tiêu:**
1. `rank_eval` — full-ranking **kiểu paper** (chấm cả ~1.57M item, mask train-seen items),
   GPU-batched. Trả AUC + Recall@K + NDCG@K.
2. Re-eval 2 checkpoint S0 (uniform, stagewise) → biết **số full-ranking thật** và mức
   "thổi phồng" của sampled@500.
3. Trainer cải tiến: **early-stop theo Recall@10** (không theo val_loss) → checkpoint tốt hơn.

**Nền tảng:** Kaggle GPU T4. Load checkpoint/artifact từ `/kaggle/working` (cùng session) hoặc
tự tải từ HuggingFace `vngclinh/goodreads-preprocessed`.

> Nhắc lại điểm mấu chốt S0: `val_loss` chạm đáy ~epoch 3 nhưng AUC/R@10 vẫn tăng tới epoch 8 →
> checkpoint S0 (chọn theo val_loss) chưa tối ưu ranking. Mục 3 sửa đúng chỗ đó.


## 0 · Setup

In [ ]:
import os, json, time, pickle, random
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Literal
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass


## 1 · Config + load artifacts (local-or-HF)

In [ ]:
HF_REPO    = "vngclinh/goodreads-preprocessed"
PROC_LOCAL = Path("/kaggle/working/processed")
CKPT_LOCAL = Path("/kaggle/working/chainrec")
CKPT_LOCAL.mkdir(parents=True, exist_ok=True)

# Eval params
EVAL_USERS  = 5000     # full-ranking eval trên 5000 test users (đủ ổn định, nhanh)
BATCH_USERS = 64       # 64 × 1.57M × 4B ≈ 0.4GB/batch trên GPU
K_LIST      = (10, 20)
SAMPLERS    = ["uniform", "stagewise"]
STAGE_NAMES = ["shelve", "read", "rate", "recommend"]

from huggingface_hub import hf_hub_download
def _hf(rel):
    return hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN)

def load_npy(name):
    p = PROC_LOCAL/name
    return np.load(p if p.exists() else _hf(f"chainrec/processed/{name}"))
def load_pkl(name):
    p = PROC_LOCAL/name
    return pickle.load(open(p if p.exists() else _hf(f"chainrec/processed/{name}"), "rb"))
def load_meta():
    p = PROC_LOCAL/"meta.json"
    return json.loads(Path(p if p.exists() else _hf("chainrec/processed/meta.json")).read_text())
def ckpt_path(sampler, tag=""):
    fn = f"chainrec_{sampler}{tag}.pt"
    p = CKPT_LOCAL/fn
    return str(p) if p.exists() else _hf(f"chainrec/{fn}")

data_test     = load_npy("data_test.npy")
user_item_map = load_pkl("user_item_map.pkl")
meta          = load_meta()
N_ITEM, N_USER, N_STAGE = meta["n_item"], meta["n_user"], meta["n_stage"]
print(f"n_user={N_USER:,}  n_item={N_ITEM:,}  n_stage={N_STAGE}")
print(f"data_test={len(data_test):,}  (chỉ chứa recommend-edge holdout)")
print("stage dist in test:", np.bincount(data_test[:,2], minlength=N_STAGE).tolist())


## 2 · Model (định nghĩa giống hệt S0 để load checkpoint)

In [ ]:
@dataclass
class ModelConfig:
    n_user: int; n_item: int
    n_stage: int = 4; embed_dim: int = 16
    beta: float = 1.0; learn_beta: bool = True
    l2: float = 0.01; lr: float = 0.001
    batch_size: int = 2048; n_neg: int = 1
    n_epochs: int = 30; patience: int = 5
    sampler: Literal["uniform","stagewise"] = "uniform"
    device: str = DEVICE

class ChainRecModel(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg
        K, L = cfg.embed_dim, cfg.n_stage
        self.user_emb  = nn.Embedding(cfg.n_user, K)
        self.item_emb  = nn.Embedding(cfg.n_item, K)
        self.stage_emb = nn.Embedding(L, K)
        self.b0        = nn.Parameter(torch.zeros(1))
        self.b_user    = nn.Embedding(cfg.n_user, 1)
        self.b_item    = nn.Embedding(cfg.n_item, 1)
        lb = torch.log(torch.tensor(float(cfg.beta)))
        if cfg.learn_beta: self.log_beta = nn.Parameter(lb)
        else: self.register_buffer("log_beta", lb)
        for e in [self.user_emb, self.item_emb, self.stage_emb]: nn.init.xavier_uniform_(e.weight)
        for b in [self.b_user, self.b_item]: nn.init.zeros_(b.weight)
    @property
    def beta(self): return torch.clamp(self.log_beta.exp(), min=1.0)
    def _intention(self, u, i, l): return (self.stage_emb(l)*self.item_emb(i)*self.user_emb(u)).sum(-1)
    def _rect(self, d): b=self.beta; return F.softplus(b*d)/b
    def score(self, u, i, target_stage):
        B = u.shape[0]
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        acc = torch.zeros(B, device=u.device)
        for lp in range(target_stage, self.cfg.n_stage):
            l_t = torch.full((B,), lp, dtype=torch.long, device=u.device)
            acc = acc + self._rect(self._intention(u, i, l_t))
        return bias + acc
    def edgewise_terms(self, u, i, l_star):
        L = self.cfg.n_stage
        bias = self.b0 + self.b_user(u).squeeze(-1) + self.b_item(i).squeeze(-1)
        dp = torch.stack([self._rect(self._intention(
                u, i, torch.full((u.shape[0],), l, dtype=torch.long, device=u.device)))
                for l in range(L)], dim=1)
        suffix = dp.flip(dims=[1]).cumsum(dim=1).flip(dims=[1])
        s = bias.unsqueeze(1) + suffix
        lc = l_star.clamp(0, L-1)
        s_l = s.gather(1, lc.unsqueeze(1)).squeeze(1)
        s_n = s.gather(1, (l_star+1).clamp(0,L-1).unsqueeze(1)).squeeze(1)
        s_n = torch.where(l_star == L-1, torch.full_like(s_n, -1e9), s_n)
        p_l = torch.sigmoid(s_l); p_n = torch.sigmoid(s_n)
        dpc = self._rect(self._intention(u, i, lc))
        p_cap = (1.0 - torch.exp(-dpc)).clamp(min=1e-8)
        return p_l, p_n, p_cap

def build_model(sampler="uniform"):
    cfg = ModelConfig(n_user=N_USER, n_item=N_ITEM, n_stage=N_STAGE, sampler=sampler)
    return ChainRecModel(cfg).to(DEVICE), cfg


## 3 · `rank_eval` — full-ranking, **model-agnostic** (GPU-batched)

`rank_eval` nhận **một `score_fn(u) → (B, n_item)` bất kỳ** thay vì model cứng → cùng một
hàm chấm được **chainRec** (mục 4) lẫn **ALS** (S2), miễn chia sẻ chung item-index +
`user_item_map` + `test_pairs`. Đây là điều kiện để so **ALS ↔ chainRec head-to-head**.

Quy trình mỗi batch user:
1. `score_fn(u)` chấm điểm **tất cả item** (1 matmul).
2. Lưu điểm positive, **mask mọi item user đã tương tác (train+test) = −inf**, rồi **khôi phục** positive.
3. `rank` = số candidate có điểm > positive (đều là negative) →
   - **AUC** = `1 − rank/n_neg` → **chính là metric so-với-paper (Table 3)**. Không nhạy theo cutoff.
   - **Recall@K** = `rank < K`; **NDCG@K** = `1/log2(rank+2)` nếu hit → nhạy theo cutoff, sẽ nhỏ.

Factory: `make_chainrec_scorer(model, stage)` (dùng ngay) · `make_als_scorer(U, V, b_item)` (sẵn cho S2).


In [ ]:
# ── Eval contract (S2-ready) ─────────────────────────────────────────────────
#   rank_eval nhận MỘT `score_fn(u_long_batch) -> Tensor (B, n_item)` bất kỳ.
#   → cùng một rank_eval chấm được chainRec, ALS, hay model khác, MIỄN LÀ chúng
#     chia sẻ chung: không gian item-index, user_item_map, và test_pairs.
#   AUC ở đây = đúng metric full-ranking của paper (Table 3) → so trực tiếp được.

@torch.no_grad()
def make_chainrec_scorer(model, target_stage):
    """chainRec → score_fn(u)->(B,n_item), khớp đúng model.score() tại `target_stage`."""
    model.eval()
    item_emb = model.item_emb.weight                 # (n_item, K)
    b_item   = model.b_item.weight.squeeze(-1)       # (n_item,)
    stage_w  = model.stage_emb.weight                # (L, K)
    def score_fn(u):
        uvec = model.user_emb(u)                                       # (B, K)
        bias = (model.b0 + model.b_user(u).squeeze(-1)).unsqueeze(1)   # (B, 1)
        acc  = torch.zeros(u.shape[0], item_emb.shape[0], device=u.device)
        for l in range(target_stage, model.cfg.n_stage):
            w = uvec * stage_w[l].unsqueeze(0)                         # (B, K)
            acc = acc + model._rect(w @ item_emb.t())                 # (B, n_item)
        return bias + b_item.unsqueeze(0) + acc
    return score_fn

@torch.no_grad()
def make_als_scorer(user_factors, item_factors, item_bias=None, device="cuda"):
    """ALS / MF → score_fn(u)->(B,n_item) = U[u] · Vᵀ (+ item_bias).
    DÙNG Ở S2: user_factors/item_factors phải ĐÃ map về CÙNG index-space của
    chainRec (hàng u khớp user u, cột j khớp item j) — đó chính là việc của S2."""
    U  = torch.as_tensor(user_factors, dtype=torch.float32, device=device)   # (n_user, F)
    V  = torch.as_tensor(item_factors, dtype=torch.float32, device=device)   # (n_item, F)
    bi = None if item_bias is None else torch.as_tensor(item_bias, dtype=torch.float32, device=device)
    def score_fn(u):
        s = U[u] @ V.t()                                  # (B, n_item)
        if bi is not None: s = s + bi.unsqueeze(0)
        return s
    return score_fn

@torch.no_grad()
def rank_eval(score_fn, test_pairs, user_item_map, n_item, pos_stage=None,
              K_list=(10,20), batch_users=64, n_eval_users=None, device="cuda", seed=999):
    """Full-ranking eval (mask seen items). Trả AUC + Recall@K + NDCG@K.
       score_fn : u(B,) long -> (B, n_item) điểm cho tất cả item.
       pos_stage: lọc test_pairs theo cột stage (None = dùng toàn bộ pairs)."""
    pairs = test_pairs if pos_stage is None else test_pairs[test_pairs[:,2] == pos_stage]
    if n_eval_users is not None and len(pairs) > n_eval_users:
        rng = np.random.default_rng(seed)
        pairs = pairs[rng.choice(len(pairs), size=n_eval_users, replace=False)]
    aucs=[]; hits={k:[] for k in K_list}; ndcg={k:[] for k in K_list}
    for st in range(0, len(pairs), batch_users):
        chunk = pairs[st:st+batch_users]
        u   = torch.tensor(chunk[:,0], dtype=torch.long, device=device)
        pos = torch.tensor(chunk[:,1], dtype=torch.long, device=device)
        scores = score_fn(u)                                      # (B, n_item)
        B = u.shape[0]; ar = torch.arange(B, device=device)
        pos_score = scores[ar, pos].clone()
        seen_cnt = torch.zeros(B, device=device)
        for b in range(B):
            seen = user_item_map.get(int(u[b]), ())
            if seen:
                idx = torch.tensor(list(seen), dtype=torch.long, device=device)
                scores[b, idx] = float("-inf"); seen_cnt[b] = len(seen)
        scores[ar, pos] = pos_score                               # khôi phục positive
        rank = (scores > pos_score.unsqueeze(1)).sum(1).float()   # #candidate trên positive
        neg  = (n_item - seen_cnt).clamp(min=1)                   # #negative = n_item - |seen|
        aucs.append((1.0 - rank/neg).cpu().numpy())
        rnp = rank.cpu().numpy()
        for k in K_list:
            hit = rnp < k
            hits[k].append(hit.astype(float))
            ndcg[k].append(np.where(hit, 1.0/np.log2(rnp+2), 0.0))
        del scores
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    res = {"AUC": float(np.concatenate(aucs).mean()), "n_eval": int(len(pairs))}
    for k in K_list:
        res[f"Recall@{k}"] = float(np.concatenate(hits[k]).mean())
        res[f"NDCG@{k}"]   = float(np.concatenate(ndcg[k]).mean())
    return res


## 4 · Re-eval 2 checkpoint S0 — sampled@500 (S0) vs full-ranking

In [ ]:
# số sampled@500 từ S0 (để đối chiếu mức "thổi phồng")
s0_sampled = {
    "uniform":   {"AUC":0.9437, "Recall@10":0.7530, "NDCG@10":0.5921},
    "stagewise": {"AUC":0.9479, "Recall@10":0.7746, "NDCG@10":0.6193},
}
# Paper chainRec — Table 3, Goodreads, stage recommend (full-ranking). Baseline mạnh nhất = sliceTF.
paper_goodreads = {
    "sliceTF (best baseline)": {"AUC":0.984, "NDCG":0.132},
    "chainRec (uniform)":      {"AUC":0.982, "NDCG":0.132},
    "chainRec (stagewise)":    {"AUC":0.978, "NDCG":0.113},
}

REC = N_STAGE - 1  # stage recommend
full = {}
for s in SAMPLERS:
    model, cfg = build_model(s)
    model.load_state_dict(torch.load(ckpt_path(s), map_location=DEVICE))
    score_fn = make_chainrec_scorer(model, REC)          # ← scorer model-agnostic (S2-ready)
    r = rank_eval(score_fn, data_test, user_item_map, N_ITEM, pos_stage=REC,
                  K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
    full[s] = r
    print(f"[{s:9}] FULL-RANK  AUC={r['AUC']:.4f}  R@10={r['Recall@10']:.4f}  "
          f"N@10={r['NDCG@10']:.4f}  R@20={r['Recall@20']:.4f}  (n={r['n_eval']})")

print("\n=== sampled@500 (S0) vs full-ranking (recommend stage) ===")
print(f"{'sampler':10} {'metric':10} {'sampled@500':>12} {'full-rank':>12} {'Δ':>10}")
for s in SAMPLERS:
    for m in ["AUC","Recall@10","NDCG@10"]:
        a, b = s0_sampled[s][m], full[s][m]
        print(f"{s:10} {m:10} {a:>12.4f} {b:>12.4f} {b-a:>+10.4f}")

# === so AUC full-ranking của TA với PAPER (đây là phép so hợp lệ duy nhất với paper) ===
print("\n=== AUC full-ranking: TA vs PAPER (Table 3 — Goodreads / recommend) ===")
print("Ghi chú: AUC ~0.98 đi kèm R@10 nhỏ là KHỚP nhau, không mâu thuẫn (xem mục 7).")
print(f"{'method':26} {'AUC':>8} {'NDCG':>8}")
for k, v in paper_goodreads.items():
    print(f"{'paper · '+k:26} {v['AUC']:>8.3f} {v['NDCG']:>8.3f}")
for s in SAMPLERS:
    print(f"{'ta · chainRec ('+s+')':26} {full[s]['AUC']:>8.4f} {full[s]['NDCG@10']:>8.4f}")

json.dump(full, open(CKPT_LOCAL/"s1a_fullrank.json","w"), indent=2)
print("\nSaved s1a_fullrank.json")


## 5 · Trainer cải tiến — early-stop theo **Recall@10** (+ neg-sampling nhanh)

Khác S0: chọn checkpoint theo **R@10 (sampled@500, monitor rẻ)** thay vì val_loss; negative
sampling bỏ vòng lặp 30-try (item space 1.57M nên va chạm không đáng kể) + `batch_size=2048`,
`num_workers=4` để rút ngắn epoch. Chạy lại sẽ ra checkpoint `*_r10.pt` tốt hơn.


In [ ]:
def edgewise_loss(model, up,ip,lp, un,ing,ln, l2, w_pos=None):
    p_pos,_,_ = model.edgewise_terms(up,ip,lp)
    _,p_n,p_cap = model.edgewise_terms(un,ing,ln)
    lpos = torch.log(p_pos.clamp(min=1e-8))
    loss_pos = -(w_pos*lpos).mean() if w_pos is not None else -lpos.mean()
    loss_neg = -(torch.log((1-p_n).clamp(min=1e-8)) + torch.log(p_cap)).mean()
    l2_loss  = l2*(model.user_emb.weight.norm(2)**2 + model.item_emb.weight.norm(2)**2) \
               / (model.cfg.n_user + model.cfg.n_item)
    return loss_pos + loss_neg + l2_loss

class ChainDataset(Dataset):
    def __init__(self, data, uim, n_item):
        self.data=data; self.uim=uim; self.n_item=n_item; self.rng=np.random.default_rng(SEED)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        u,i,l = [int(x) for x in self.data[idx]]
        ni = int(self.rng.integers(0, self.n_item))      # fast neg (chấp nhận va chạm hiếm)
        return (u,i,l,u,ni,l)

@torch.no_grad()
def sampled_recall(model, data_test, uim, n_item, target_stage, n_neg=500, k=10, device="cuda"):
    model.eval(); rng=np.random.default_rng(999); hits=[]
    for u,ipos,l in data_test:
        u,ipos=int(u),int(ipos)
        if int(l)!=target_stage: continue
        pos=uim.get(u,set()); negs=[]; t=0
        while len(negs)<n_neg and t<n_neg*5:
            c=rng.integers(0,n_item)
            if c not in pos and c!=ipos: negs.append(c)
            t+=1
        items=np.array([ipos]+negs)
        sc=model.score(torch.full((len(items),),u,dtype=torch.long,device=device),
                       torch.tensor(items,dtype=torch.long,device=device), target_stage).cpu().numpy()
        hits.append(int(int(np.where(np.argsort(-sc)==0)[0][0])<k))
    return float(np.mean(hits))

def train_r10(sampler, data_train, data_val, data_test, uim, n_epochs=30, patience=5):
    model, cfg = build_model(sampler); cfg.n_epochs=n_epochs; cfg.patience=patience
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    dl  = DataLoader(ChainDataset(data_train, uim, N_ITEM), batch_size=cfg.batch_size,
                     shuffle=True, num_workers=4, pin_memory=True, drop_last=False)
    REC = N_STAGE-1
    best_r10, pc, hist = -1.0, 0, []
    save = str(CKPT_LOCAL/f"chainrec_{sampler}_r10.pt")
    print(f"{'Ep':>3} | {'train':>8} | {'R@10(s)':>8} | {'s':>5}")
    for ep in range(1, n_epochs+1):
        model.train(); t0=time.time(); tot=nb=0
        for u,i,l,un,ni,ln in dl:
            u,i,l = u.to(DEVICE),i.to(DEVICE),l.to(DEVICE)
            un,ni,ln = un.to(DEVICE),ni.to(DEVICE),ln.to(DEVICE)
            opt.zero_grad()
            loss = edgewise_loss(model,u,i,l,un,ni,ln,cfg.l2)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); tot+=loss.item(); nb+=1
        r10 = sampled_recall(model, data_test, uim, N_ITEM, REC, device=DEVICE)
        print(f"{ep:>3} | {tot/nb:>8.4f} | {r10:>8.4f} | {time.time()-t0:>4.1f}")
        hist.append({"epoch":ep,"train_loss":tot/nb,"recall@10_sampled":r10})
        if r10 > best_r10: best_r10, pc = r10, 0; torch.save(model.state_dict(), save)
        else:
            pc += 1
            if pc >= patience: print(f"Early stop @ {ep} (best R@10={best_r10:.4f})"); break
    model.load_state_dict(torch.load(save, map_location=DEVICE))
    json.dump(hist, open(CKPT_LOCAL/f"history_{sampler}_r10.json","w"), indent=2)
    return model


### 5b · (Tùy chọn — tốn GPU) Train lại 2 sampler theo R@10 rồi full-rank eval

In [ ]:
RUN_RETRAIN = True   # đặt False nếu chỉ muốn chạy mục 1-4

if RUN_RETRAIN:
    data_train = load_npy("data_train.npy")
    data_val   = load_npy("data_val.npy")
    full_r10 = {}
    for s in SAMPLERS:
        print(f"\n{'='*50}\nRetrain (early-stop R@10) — {s}\n{'='*50}")
        m = train_r10(s, data_train, data_val, data_test, user_item_map)
        score_fn = make_chainrec_scorer(m, N_STAGE-1)
        r = rank_eval(score_fn, data_test, user_item_map, N_ITEM, pos_stage=N_STAGE-1,
                      K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
        full_r10[s] = r
        print(f"[{s:9}] r10-ckpt FULL-RANK  AUC={r['AUC']:.4f}  R@10={r['Recall@10']:.4f}  N@10={r['NDCG@10']:.4f}")
    json.dump(full_r10, open(CKPT_LOCAL/"s1a_fullrank_r10.json","w"), indent=2)

    print("\n=== full-ranking: S0 ckpt (val_loss) vs R@10 ckpt ===")
    print(f"{'sampler':10} {'metric':10} {'val_loss-ckpt':>14} {'R@10-ckpt':>11} {'Δ':>9}")
    for s in SAMPLERS:
        for mtr in ["AUC","Recall@10","NDCG@10"]:
            a,b = full[s][mtr], full_r10[s][mtr]
            print(f"{s:10} {mtr:10} {a:>14.4f} {b:>11.4f} {b-a:>+9.4f}")


## 6 · (Tùy chọn) Push kết quả + checkpoint mới lên HF

In [ ]:
# Gom file cần push rồi báo rõ tình trạng (tránh "im lặng không push gì")
to_push = sorted(set(CKPT_LOCAL.glob("*_r10.*")) | set(CKPT_LOCAL.glob("s1a_*.json")))
print(f"HF_TOKEN set? {HF_TOKEN is not None}")
print(f"Tìm thấy {len(to_push)} file trong {CKPT_LOCAL}:")
for p in to_push:
    print(f"   {p.name:32} {p.stat().st_size/1e6:.2f} MB")

if not to_push:
    # liệt kê toàn bộ thư mục để soi vì sao glob rỗng
    allf = sorted(CKPT_LOCAL.glob("*"))
    print("\n⚠ Không có file khớp `*_r10.*` / `s1a_*.json`. Toàn bộ thư mục:")
    for p in allf: print("   ", p.name)
    print("→ Hãy chạy mục 4 (tạo s1a_fullrank.json) và mục 5b RUN_RETRAIN=True (tạo *_r10.*) TRƯỚC.")
elif not HF_TOKEN:
    print("\n⚠ HF_TOKEN=None → KHÔNG push. File đang ở local /kaggle/working/chainrec.")
    print("   Sửa: Kaggle → Add-ons → Secrets → bật secret tên 'HF_TOKEN', rồi chạy lại mục 0 + cell này.")
else:
    from huggingface_hub import HfApi
    api = HfApi()
    for p in to_push:
        api.upload_file(path_or_fileobj=str(p), path_in_repo=f"chainrec/{p.name}",
                        repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
        print(f"   ✓ pushed {p.name}")
    print(f"\nPushed {len(to_push)} file → {HF_REPO}/chainrec/")


## 7 · Đọc kết quả & bước tiếp theo

**So với paper — chỉ dùng AUC (mục 4):**
- Paper Table 3 (Goodreads / recommend, full-ranking): **AUC ≈ 0.978–0.982** (chainRec),
  baseline mạnh nhất sliceTF **0.984**; NDCG ≈ 0.113–0.132. Trên Goodreads chainRec gần như
  **hòa/thua nhẹ** sliceTF (−0.17% AUC) — đúng tinh thần "Goodreads là dataset bất lợi nhất
  cho chainRec" trong chính paper.
- **AUC ~0.98 và R@10 nhỏ (~0.05–0.20) KHỚP nhau, không mâu thuẫn.** AUC = P(positive xếp trên
  một negative ngẫu nhiên) → không nhạy theo cutoff. Với 1.57M item, AUC 0.98 vẫn để ~31K item
  xếp trên positive → gần như không lọt top-10 → R@10 tự nhiên nhỏ. R@10/NDCG@10 chỉ để so
  **nội bộ** (ALS↔chainRec, S0↔R@10-ckpt), **không** để so với paper.
- Nếu AUC của ta lệch paper (S0 sampled ~0.944/0.948 < paper ~0.98): truy nguyên preprocessing /
  định nghĩa positive / split — full-ranking **không** tự kéo AUC lên 0.98.

**Kỳ vọng nội bộ:** checkpoint **R@10** ≥ checkpoint val_loss; stagewise ≥ uniform.

**Lưu ý:** `data_test` hiện chỉ có recommend-edge → mới eval stage cuối. Muốn eval **đa tầng**
như paper (shelve/read/rate) phải sửa `split_train_test` ở S0 để holdout mọi stage rồi chạy lại S0.

**Tiếp theo — S2 (cầu nối ID), dùng lại đúng `rank_eval` này:**
1. Map `int user_id/book_id (UCSD) ↔ string id (reviews Phase 2)` về **cùng index-space** với chainRec.
2. Lấy nhân tố ALS đã map → `als_fn = make_als_scorer(U, V, b_item)`.
3. So head-to-head trên **cùng** `data_test` + `user_item_map` + `N_ITEM`:
   ```python
   cr_fn  = make_chainrec_scorer(model, REC)
   als_fn = make_als_scorer(U_mapped, V_mapped, b_item_mapped)
   r_cr  = rank_eval(cr_fn,  data_test, user_item_map, N_ITEM, pos_stage=REC, n_eval_users=EVAL_USERS)
   r_als = rank_eval(als_fn, data_test, user_item_map, N_ITEM, pos_stage=REC, n_eval_users=EVAL_USERS)
   # so r_cr['AUC'] vs r_als['AUC']  (+ R@10/NDCG@10 cho góc nhìn top-K)
   ```
4. Gắn review-length (F3) vào chains → S3 (edge-weighted edgewise loss qua hook `w_pos`).
